# Multi-Hazard Risk & Emergency Routing — Wayanad District, Kerala
End-to-end demo: **Satellite data → indicator/model → recommendation**.

Before running: authenticate once (`earthengine authenticate`) and set your Earth Engine Cloud project below.

In [ ]:
import os, sys, logging
sys.path.insert(0, os.path.abspath('.'))
os.environ.setdefault('GEE_PROJECT', 'your-gee-project-id')   # <-- edit
import config
from src import pipeline, gee_init
from src.utils import setup_logging
setup_logging()
print('Project:', os.environ['GEE_PROJECT'])

## 1. Run the full pipeline
Flood (Sentinel-1 + Otsu), landslide (FR-style), fire (MODIS/VIIRS density), exposure (Dynamic World + roads) → AHP + entropy MCDM → classes, hotspots, least-risk routes → `index.html`.
Set `FROM_CACHE = True` to rebuild from `outputs/` without Earth Engine.

In [ ]:
FROM_CACHE = False
try:
    results = pipeline.run(from_cache=FROM_CACHE)
except gee_init.GEEInitError as err:
    raise SystemExit(str(err))
summary = results['summary']

## 2. Landslide validation gate (July 2024 Mundakkai–Chooralmala)

In [ ]:
import json
val = summary['validation']
print('Headline gate:', 'PASS' if val['headline_passed'] else 'FAIL')
for r in val['results']:
    print(f"[{'PASS' if r['passed'] else 'FAIL'}] {r['label']:9s} ({r['lat']:.3f}, {r['lon']:.3f})  point={r['point_class_name']}  modal={r['zone_modal_class']}  High/VH share={r['zone_fraction_high_or_very_high']:.0%}")
    if 'diagnosis' in r: print('   ->', r['diagnosis'])

## 3. MCDM weights (AHP vs entropy vs combined vs CRITIC) and consistency ratio

In [ ]:
import pandas as pd
print('AHP CR =', round(summary['ahp']['cr'], 4), '(must be < 0.1)')
pd.DataFrame(summary['weight_table']).T.round(4)

## 4. Risk-class area statistics

In [ ]:
pd.DataFrame(summary['stats'])

## 5. Shortest vs least-risk routes

In [ ]:
pd.DataFrame(summary['routes'])[['origin','destination_type','destination','shortest_km','least_risk_km','distance_increase_pct','shortest_mean_risk','least_risk_mean_risk','mean_risk_reduction_pct']]

## 6. Interactive map
The standalone map is written to `index.html` (open it in any browser).

In [ ]:
from IPython.display import IFrame
IFrame(src='index.html', width='100%', height=650)

## 7. Optional: explore the Earth Engine layers with geemap
Run after the pipeline to inspect individual layers on a live map.

In [ ]:
import geemap, ee
aoi, _, _ = gee_init.get_study_area()
layers = pipeline.compute_layers(aoi, None)
Map = geemap.Map(center=[config.MAP_CENTER[0], config.MAP_CENTER[1]], zoom=config.MAP_ZOOM)
for name, key in [('Flood','flood'), ('Landslide','landslide'), ('Fire','fire'), ('Exposure','exposure')]:
    Map.addLayer(layers[key]['image'], {'min': 0, 'max': 1, 'palette': list(config.RISK_RAMP)}, name, False)
Map.addLayer(ee.Image().paint(aoi, 0, 2), {'palette': 'black'}, 'Study area')
Map